# Compile And Runtime Tests

This notebook is the unit/integration test suite for the TVM compile path. It uses plain Python `assert` statements so each cell fails loudly if a required step breaks.

The tests are intentionally small and deterministic. They create a simple computer-vision classifier that scores red, green, and blue images, then verify that TVM can compile and run it through both Python and C++ runtimes.

## What This Tests

This notebook validates the practical workflow the project cares about:

1. Build a deterministic image classifier in PyTorch.
2. Compile that PyTorch model for the `x86_64` target profile.
3. Run the compiled PyTorch artifacts with TVM's Python graph executor.
4. Export the same model to ONNX.
5. Compile the ONNX model for `x86_64`.
6. Run the ONNX artifacts with the Python graph executor.
7. Build the C++ graph executor program.
8. Run the ONNX artifacts from C++ and verify the prediction.

The model and test image are generated locally. No network download or pretrained-weight cache is required.

In [1]:
from __future__ import annotations

import os
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

EXAMPLES = ROOT / 'examples' / 'python'
if str(EXAMPLES) not in sys.path:
    sys.path.insert(0, str(EXAMPLES))

import tvm  # noqa: F401
from tvm_prep.compile import build_and_export
from tvm_prep.model_zoo import LoadedModel, load_onnx_model
from tvm_prep.preprocess import load_imagenet_image
from tvm_prep.runtime import run_graph_executor, topk
from tvm_prep.targets import get_target_profile

TEST_ROOT = Path(tempfile.mkdtemp(prefix='tvm-prep-tests-'))
RED_IMAGE = TEST_ROOT / 'red.png'
Image.new('RGB', (256, 256), (255, 0, 0)).save(RED_IMAGE)

print('Repository:', ROOT)
print('Temporary test directory:', TEST_ROOT)
print('TVM version:', tvm.__version__)

Repository: /workspaces/TVM-Prep-guide
Temporary test directory: /tmp/tvm-prep-tests-4yi0vg5h
TVM version: 0.19.0


## Test Model

The model averages the red, green, and blue channels and applies a fixed linear classifier. It acts like a tiny trained image classifier: a red image should produce class `0`, a green image class `1`, and a blue image class `2`.

This is enough to test TVM import, compilation, artifact export, and runtime behavior without downloading a large pretrained model.

In [2]:
class RGBMeanClassifier(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.classifier = torch.nn.Linear(3, 3, bias=False)
        with torch.no_grad():
            self.classifier.weight.copy_(
                torch.tensor(
                    [
                        [4.0, -1.0, -1.0],
                        [-1.0, 4.0, -1.0],
                        [-1.0, -1.0, 4.0],
                    ],
                    dtype=torch.float32,
                )
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        channel_means = x.mean(dim=(2, 3))
        return self.classifier(channel_means)


def make_pytorch_model() -> LoadedModel:
    torch.manual_seed(0)
    torch.set_grad_enabled(False)
    return LoadedModel(
        name='rgb_mean_classifier',
        frontend='pytorch',
        model=RGBMeanClassifier().eval(),
        input_name='input0',
        input_shape=(1, 3, 224, 224),
        labels=['red', 'green', 'blue'],
    )


PYTORCH_MODEL = make_pytorch_model()
print(PYTORCH_MODEL)

LoadedModel(name='rgb_mean_classifier', frontend='pytorch', model=RGBMeanClassifier(
  (classifier): Linear(in_features=3, out_features=3, bias=False)
), input_name='input0', input_shape=(1, 3, 224, 224), input_dtype='float32', labels=['red', 'green', 'blue'])


## Test Helpers

These helper functions keep the assertions readable. They compile to the `x86_64` target profile, verify the expected artifact files exist, and check that the compiled model predicts `red` for a generated red image.

In [3]:
def compile_for_x86(loaded: LoadedModel, output_root: Path) -> Path:
    artifact_dir = build_and_export(loaded, get_target_profile('x86_64'), output_root, opt_level=1)
    for name in ['metadata.json', 'model.json', 'model.params', 'model.so']:
        assert (artifact_dir / name).exists(), f'Missing artifact: {name}'
    return artifact_dir


def assert_python_runtime_predicts_red(artifact_dir: Path) -> None:
    image = load_imagenet_image(RED_IMAGE, input_shape=(1, 3, 224, 224), normalize=False)
    output = run_graph_executor(artifact_dir, image)
    predictions = topk(output, ['red', 'green', 'blue'], k=3)
    print(predictions)
    assert predictions[0][0] == 0
    assert predictions[0][2] == 'red'
    assert predictions[0][1] > predictions[1][1]


def export_to_onnx(loaded: LoadedModel, onnx_path: Path) -> LoadedModel:
    example_input = torch.zeros(loaded.input_shape, dtype=torch.float32)
    torch.onnx.export(
        loaded.model,
        example_input,
        onnx_path,
        input_names=[loaded.input_name],
        output_names=['logits'],
        opset_version=13,
        do_constant_folding=True,
    )
    imported = load_onnx_model(onnx_path, loaded.input_name, loaded.input_shape)
    return LoadedModel(
        name=imported.name,
        frontend=imported.frontend,
        model=imported.model,
        input_name=imported.input_name,
        input_shape=imported.input_shape,
        input_dtype=imported.input_dtype,
        labels=loaded.labels,
    )

## Test 1: PyTorch To TVM To Python Runtime

This test verifies the direct PyTorch frontend path. TVM traces the PyTorch module, imports it to Relay, compiles it for x86_64, exports artifacts, and runs those artifacts through Python.

In [4]:
pytorch_artifacts = compile_for_x86(PYTORCH_MODEL, TEST_ROOT / 'artifacts-pytorch')
assert_python_runtime_predicts_red(pytorch_artifacts)
print('PASS: PyTorch model compiled and ran with Python runtime')

One or more operators have not been tuned. Please tune your model for better performance. Use DEBUG logging level to see more details.


[(0, 0.9867032766342163, 'red'), (2, 0.006648354232311249, 'blue'), (1, 0.006648354232311249, 'green')]
PASS: PyTorch model compiled and ran with Python runtime


## Test 2: ONNX To TVM To Python Runtime

This test exports the same model to ONNX, imports that ONNX file into TVM, compiles it for x86_64, and runs the compiled output with Python. This validates the file-based model path used by many real computer-vision workflows.

In [5]:
ONNX_MODEL = export_to_onnx(PYTORCH_MODEL, TEST_ROOT / 'rgb_mean_classifier.onnx')
onnx_artifacts = compile_for_x86(ONNX_MODEL, TEST_ROOT / 'artifacts-onnx')
assert_python_runtime_predicts_red(onnx_artifacts)
print('PASS: ONNX model compiled and ran with Python runtime')

/home/vscode/.local/lib/python3.11/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.Op.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
/home/vscode/.local/lib/python3.11/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.OnnxFunction.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()


[(0, 0.9867032766342163, 'red'), (2, 0.006648354232311249, 'blue'), (1, 0.006648354232311249, 'green')]
PASS: ONNX model compiled and ran with Python runtime


## Test 3: ONNX Artifacts With The C++ Runtime

The C++ runner loads the same artifact directory as the Python runtime. This cell builds the runner with CMake/Ninja, runs it on the ONNX-compiled artifacts, and checks that the first prediction is the red class.

In [6]:
cpp_build_dir = TEST_ROOT / 'cpp-build'
subprocess.run(
    ['cmake', '-S', str(ROOT / 'examples/cpp'), '-B', str(cpp_build_dir), '-G', 'Ninja'],
    check=True,
    cwd=ROOT,
)
subprocess.run(['cmake', '--build', str(cpp_build_dir)], check=True, cwd=ROOT)

runner = cpp_build_dir / 'run_tvm_graph'
env = os.environ.copy()
env['LD_LIBRARY_PATH'] = f"/opt/tvm/build:{env.get('LD_LIBRARY_PATH', '')}"
result = subprocess.run(
    [
        str(runner),
        '--artifact-dir',
        str(onnx_artifacts),
        '--image',
        str(RED_IMAGE),
        '--no-normalize',
    ],
    check=True,
    text=True,
    capture_output=True,
    env=env,
)

print(result.stdout)
first_line = result.stdout.splitlines()[0]
assert '1: id=0' in first_line
assert 'red' in first_line
print('PASS: ONNX artifacts ran successfully with C++ runtime')

-- The CXX compiler identification is GNU 12.2.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Configuring done
-- Generating done
-- Build files have been written to: /tmp/tvm-prep-tests-4yi0vg5h/cpp-build
[1/2] Building CXX object CMakeFiles/run_tvm_graph.dir/run_tvm_graph.cpp.o
In file included from /opt/tvm/include/tvm/runtime/container/base.h:28,
                 from /opt/tvm/include/tvm/runtime/container/string.h:29,
                 from /opt/tvm/include/tvm/runtime/module.h:31,
                 from /workspaces/TVM-Prep-guide/examples/cpp/run_tvm_graph.cpp:2:
/opt/tvm/include/tvm/runtime/logging.h:594: warning: "LOG" redefined
  594 | #define LOG(level) LOG_##level
      | 
In file included from /opt/tvm/3rdparty/dmlc-core/include/dmlc/io.h:15,
                 from /opt/tvm/include/tvm/runtime/module.h:29:
/opt/

## Command-Line Validation

To run this notebook as an automated validation command from the repository root:

```bash
jupyter nbconvert --to notebook --execute notebooks/01_compile_runtime_tests.ipynb \
  --output /tmp/01_compile_runtime_tests.executed.ipynb \
  --ExecutePreprocessor.timeout=900
```

If the command exits successfully, the Python and C++ TVM paths both worked.